In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import jaccard_score
from sklearn.preprocessing import binarize
import numpy as np
from itertools import combinations
from tqdm import tqdm

# Load CSV
df = pd.read_csv('LLM_Insight_NFL_Assistant.csv')

df.head(5)

,Question_TimeStamp,Answer_TimeStamp,Question_Category,Question,Answer_Text
0,session_aecb9f26_6240,session_aecb9f26_6240,Cat_1,Our team is preparing to face the Ravens next ...,Here's a tactical scouting report on the Raven...
1,session_7d41fbca_1981,session_7d41fbca_1981,Cat_1,Our team is preparing to face the Ravens next ...,Here's a scouting report on the Ravens' offens...
2,session_e25df69d_3541,session_e25df69d_3541,Cat_1,Our team is preparing to face the Ravens next ...,"Utkarsh, To prepare for facing the Ravens, her..."
3,session_c4bb3c35_7456,session_c4bb3c35_7456,Cat_1,Our team is preparing to face the Ravens next ...,"Based on the provided play-by-play data, here ..."
4,session_cf091fba_3770,session_cf091fba_3770,Cat_1,Our team is preparing to face the Ravens next ...,Here's a tactical breakdown of what to expect ...


In [3]:
# Install first (if not already installed):
# pip install sentence-transformers

from sentence_transformers import SentenceTransformer, util

# Load a pre-trained SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight and fast

# Define two sample texts to compare
text1 = "The assistant is responsible for verifying attendance and working hours."
text2 = "The duties involve ensuring compliance with schedules and managing attendance."

# Encode both texts into dense embeddings
embedding1 = model.encode(text1, convert_to_tensor=True)
embedding2 = model.encode(text2, convert_to_tensor=True)

# Compute cosine similarity
cosine_score = util.cos_sim(embedding1, embedding2)

# Print the result
print(f"Semantic Similarity Score (Cosine): {cosine_score.item():.4f}")


Semantic Similarity Score (Cosine): 0.6351


In [4]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from itertools import combinations

# Load SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Drop NaNs or empty answers
df = df.dropna(subset=['Answer_Text', 'Question_Category'])
df['Answer_Text'] = df['Answer_Text'].astype(str).str.strip()
df = df[df['Answer_Text'] != '']

# Results container
results = []

# Group by question
for question, group in df.groupby('Question_Category'):
    answers = group['Answer_Text'].tolist()
    indices = group.index.tolist()

    # Generate SBERT embeddings
    embeddings = model.encode(answers, convert_to_tensor=True)

    # Calculate cosine similarity matrix
    cosine_sim = util.cos_sim(embeddings, embeddings)

    # Loop over all unique pairs
    for i, j in combinations(range(len(answers)), 2):
        cosine_score = cosine_sim[i][j].item()
        results.append({
            'Question_Category': question,
            'Answer_1': answers[i],
            'Answer_2': answers[j],
            'Index_1': indices[i],
            'Index_2': indices[j],
            'SBERT_Cosine_Similarity': cosine_score
        })

# Convert to DataFrame
similarity_df = pd.DataFrame(results)

# Optional: Save to CSV
#similarity_df.to_csv('sbert_answer_text_similarity_llm.csv', index=False)


In [5]:
similarity_df

,Question_Category,Answer_1,Answer_2,Index_1,Index_2,SBERT_Cosine_Similarity
0,Cat_1,Here's a tactical scouting report on the Raven...,Here's a scouting report on the Ravens' offens...,0,1,0.920682
1,Cat_1,Here's a tactical scouting report on the Raven...,"Utkarsh, To prepare for facing the Ravens, her...",0,2,0.888781
2,Cat_1,Here's a tactical scouting report on the Raven...,"Based on the provided play-by-play data, here ...",0,3,0.887726
3,Cat_1,Here's a tactical scouting report on the Raven...,Here's a tactical breakdown of what to expect ...,0,4,0.888938
4,Cat_1,Here's a tactical scouting report on the Raven...,"Based on the fragmented play-by-play data, her...",0,5,0.836735
...,...,...,...,...,...,...
105,Cat_2,Here's a Player-Level Behavior Summary for the...,Here's a Player-Level Behavior Summary for the...,17,19,0.910975
106,Cat_2,Here's a Player-Level Behavior Summary for the...,Lamar Jackson is the centerpiece of the Ravens...,17,21,0.698446
107,Cat_2,Here's a Player-Level Behavior Summary for the...,Here's a Player-Level Behavior Summary for the...,18,19,0.910975
108,Cat_2,Here's a Player-Level Behavior Summary for the...,Lamar Jackson is the centerpiece of the Ravens...,18,21,0.698446
